# Day 2 — Tool/Memory Validation, Multi-Hop Reasoning & Failure-Path Testing

**Module 6 · Agentic RAG Testing**

---

## What we'll cover today

| # | Topic | Why it matters |
|---|---|---|
| 1 | Memory validation across hops | Same boundary-value problem as Module 5's chunk-boundary bug, relocated to conversation memory |
| 2 | Hard negative: right facts, wrong combination | A failure mode per-fact faithfulness checks structurally cannot catch |
| 3 | Failure-path testing | What should happen when a hop comes up empty — the exact Klarna gap, testable |
| 4 | Extending the coverage matrix | New columns: `reasoning_chain_break`, `premature_stop`, `ungraceful_failure` |

**Estimated time:** 60 minutes
**Run order:** top to bottom. Every cell runs offline.

---

> **Where we are in the course**
> Day 1 built a working, traced 2-hop loop and named 4 new failure modes that only exist in multi-step retrieval.
> Module 5 Day 2 taught boundary value analysis on chunk size; Module 5 Day 3 taught hard negatives for faithfulness.
> Today both come back, pointed at the planner's memory and its reasoning across hops instead of a single retrieval.

---
## Memory validation: did the early fact survive?

A multi-hop chain passes facts from early hops forward into the final generation step. Module 4 Day 4 taught you to boundary-test a context window; this is the same boundary, relocated: **does the fact from hop 1 still make it into the final answer, or does it get dropped or corrupted by the time hop 3 generates a response?**

> **Plain English:** this is the chunk-boundary bug from Module 5 Day 2, except the "chunk" is the running memory of a multi-hop conversation instead of a document. The boundary is wherever your agent truncates history — and it can split a needed fact out exactly the same way a 500-token chunk boundary could.

In [ ]:
def facts_present_in_answer(answer: str, required_facts: list[str]) -> dict:
    missing = [f for f in required_facts if f not in answer]
    return {"missing": missing, "passed": len(missing) == 0}

required = ["WidgetPro 3000"]  # the hop-1 conclusion (WHICH product) that must survive into the final answer

# Memory intact: hop 1's conclusion is reflected in the final answer
answer_with_memory = "WidgetPro 3000's cancellation fee is $0, since it replaced WidgetPro 2000 in 2023."
# Memory dropped: the final answer only reflects the LAST hop's number, hop 1's product identity got truncated away
answer_memory_dropped = "The cancellation fee is $0."

for label, answer in [("MEMORY INTACT", answer_with_memory), ("MEMORY DROPPED", answer_memory_dropped)]:
    result = facts_present_in_answer(answer, required)
    print(f"[{label}] -> passed={result['passed']}  missing={result['missing']}")

---
## Hard negative — right facts, wrong combination (`reasoning_chain_break`)

This is the failure mode Module 5's single-hop metrics structurally cannot catch, because both retrieved facts are individually faithful to their source — the bug is in how they were *combined*.

In [ ]:
hop1_fact = "WidgetPro 2000 was discontinued in 2023 and replaced by WidgetPro 3000."
hop2_facts = [
    "WidgetPro 3000's cancellation fee is $0.",
    "WidgetPro 2000's cancellation fee was $50.",
]

# Hard negative: the agent answers with the OLD product's fee instead of the new one's —
# every individual fact is true and grounded; the COMBINATION is wrong.
reasoning_break_answer = "The cancellation fee for the product that replaced WidgetPro 2000 is $50."
correct_answer = "The cancellation fee for the product that replaced WidgetPro 2000 (WidgetPro 3000) is $0."

def check_correct_product_fee(answer: str) -> bool:
    # The question asks about the REPLACEMENT product — the answer must cite WidgetPro 3000's fee, not 2000's.
    return "$0" in answer and "WidgetPro 3000" in answer

for label, answer in [("HARD NEGATIVE (should fail)", reasoning_break_answer), ("CORRECT (should pass)", correct_answer)]:
    print(f"[{label}] -> passed={check_correct_product_fee(answer)}")

print()
print("A faithfulness check alone would PASS the hard negative above — '$50' really is in the")
print("retrieved context. This is exactly why agentic RAG needs reasoning-level checks on top of")
print("Module 5's faithfulness/groundedness checks, not instead of them.")

---
## Failure-path testing: the hop that comes up empty

What should the agent do when a hop finds nothing relevant?

- **Graceful** — admits the gap: *"I found that WidgetPro 3000 replaced WidgetPro 2000, but I don't have its cancellation fee on file."*
- **Ungraceful** — confidently invents a number to fill the gap.

This hard negative is aimed squarely at the failure pattern behind Klarna's "complex cases dropped in quality" from Day 1 — an agent that can't find the next fact should say so, not fabricate one to keep the chain moving.

In [ ]:
def empty_hop_response_is_graceful(response: str) -> bool:
    hedge_phrases = ["don't have", "couldn't find", "no information", "not available"]
    return any(phrase in response.lower() for phrase in hedge_phrases)

graceful   = "I found that WidgetPro 3000 replaced WidgetPro 2000, but I don't have its cancellation fee on file."
ungraceful = "The cancellation fee for WidgetPro 3000 is $25."   # invented — this hop found nothing

print(f"graceful   -> {empty_hop_response_is_graceful(graceful)}")
print(f"ungraceful -> {empty_hop_response_is_graceful(ungraceful)}")
print()
print("'ungraceful' is the more dangerous of the two precisely because it LOOKS like a normal,")
print("confident answer — nothing about its surface form signals that anything went wrong.")

---
## Extending the coverage matrix

Module 5 Day 2 added retrieval columns to Module 4 Day 4's matrix. Today adds the agentic ones.

In [ ]:
from collections import Counter

annotated_cases = [
    {"id": "single-hop-01",   "category": "single_hop_qa", "failure_mode": "hallucination"},
    {"id": "multi-hop-01",    "category": "multi_hop_qa",  "failure_mode": "reasoning_chain_break"},
    {"id": "multi-hop-02",    "category": "multi_hop_qa",  "failure_mode": "ungraceful_failure"},
]

categories    = sorted({"single_hop_qa", "multi_hop_qa"})
failure_modes = sorted({"hallucination", "reasoning_chain_break", "premature_stop", "ungraceful_failure"})
counts        = Counter((c["category"], c["failure_mode"]) for c in annotated_cases)

header = " " * 16 + "".join(f"{fm:<24}" for fm in failure_modes)
print(header)
for cat in categories:
    row = f"{cat:<16}" + "".join(f"{counts[(cat, fm)]:<24}" for fm in failure_modes)
    print(row)

print()
print("multi_hop_qa x premature_stop reads 0 — a named, visible gap (Day 1's Try-It-Yourself")
print("exercise built exactly this bug; nothing in this dataset would catch it yet).")

---
## Try It Yourself

1. Write a third memory-validation case where the final answer reflects hop 1's fact but **drops hop 2's entirely**. Does `facts_present_in_answer()` catch it the same way it caught the fully-dropped case above?
2. Add a `premature_stop` row to `annotated_cases` so that cell in the matrix is no longer `0` — base it on the bug you built in Day 1's Try-It-Yourself Part 1.
3. Write your own "right facts, wrong combination" hard negative in a domain other than product fees (e.g. dates, locations, prices). What made it easy or hard to construct compared to the WidgetPro example?

Exercise file: [`exercises/02_tool_memory_reasoning_exercise.md`](../exercises/02_tool_memory_reasoning_exercise.md)

---
## Summary

### What we built today
- A memory-validation check that catches facts dropped between hops — the conversational version of Module 5's chunk-boundary bug
- A `reasoning_chain_break` hard negative that a per-fact faithfulness check would have missed entirely
- A graceful-vs-ungraceful failure check aimed at the exact gap behind the Klarna incident
- A coverage matrix extended with `reasoning_chain_break`, `premature_stop`, and `ungraceful_failure`

### The thread through this whole module
Every check built across these two days reused a Module 4/5 technique and pointed it one level up: equivalence partitioning -> hop count, boundary value analysis -> conversation memory, hard negatives -> reasoning combination and graceful failure, coverage matrix -> agentic failure modes. Same mindset, new surface — same habit this course keeps building.

**Next:** Module 7 — AI Agents Testing with DeepEval, where these hand-rolled checks become formal, reusable metrics (task completion, tool correctness, argument correctness) and the agent gains real tool-calling, not just retrieval.

---